# Réseau time series — GRU bidirectionnel + statique (LightGBM + physique)
`f(t) = météo + schedules d'usage + consignes + calendrier + écarts` → GRU **bidirectionnel**, fusionné avec `s = 4 prédictions LightGBM + 42 features physiques + 2 présence d'équipement` → MLP → conso(t), 4 cibles.

Modèle **non causal** : pour l'heure `t` il lit toute la semaine, passé et futur. C'est un **simulateur** — reconstruire la courbe d'un bâtiment jamais mesuré sur une année météo connue — et non un prédicteur temps réel : aucune consommation passée n'entre en entrée.

Deux choix sont pilotables en tête de notebook, et documentés par les A/B qui les ont fixés :
- **`TETE`** — `'additive'` par défaut (le réseau prédit des kWh). La variante `'multiplicative'` (forme × niveau annuel LightGBM) **dégrade le chauffage**, cf. docstring de `LoadNet`.
- **`DROP_CONSIGNES`** (cellule `s`) — les 6 colonnes de consigne sortent de `s` : elles sont déjà dans `f(t)` heure par heure. Elles restent dans `X_47features.parquet` pour LightGBM, qui n'a pas de série temporelle.

Voir `plan/amelioration_timeseries_net.md` pour l'historique des essais et des mesures.

**Partie 1** : squelette validé sur données factices. **Partie 2** : vraies données.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA   = Path().resolve().parent.parent / 'data' / 'processed'
TS_BASE = ('https://oedi-data-lake.s3.amazonaws.com/nrel-pds-building-stock/'
           'end-use-load-profiles-for-us-building-stock/2025/resstock_amy2018_release_1/'
           'timeseries_individual_buildings/by_state/upgrade=0')

N_STATIC = 4     # taille de s pour la Partie 1 (données factices) ; en Partie 2 -> S.shape[1]
N_OUT    = 4     # cibles au pas de temps
HIDDEN   = 128   # 64 -> 128 : mesuré le 31/07 en même temps que le passage au bidirectionnel
SEQ_LEN  = 168   # fenêtre = 1 semaine horaire

# Forme de la tête de sortie. 'additive' = le réseau prédit directement des kWh (modèle du
# 31/07) ; 'multiplicative' = il prédit une forme sans dimension, multipliée par le niveau
# annuel LightGBM. Testé le 04/08 : la multiplicative DÉGRADE le chauffage (voir LoadNet).
# À re-tester quand le climat sera dans s — pas avant.
TETE = 'additive'
assert TETE in ('additive', 'multiplicative')

In [ ]:
# softplus(0.5413) = 1 : biais de sortie qui fait démarrer la forme à 1, donc pile sur le
# niveau annuel. N'a de sens qu'en tête multiplicative.
BIAIS_UN = float(np.log(np.e - 1))


class LoadNet(nn.Module):
    """GRU bidirectionnel sur f(t) + vecteur statique -> conso(t). Sortie contrainte >= 0.

    NON CAUSAL : pour l'heure t, le réseau lit toute la semaine, passé ET futur. Légitime ici
    car on simule sur une année météo connue d'avance, pas en prévision temps réel.

    Mesuré le 31/07 (503 bâtiments, mêmes données/graine, S = 52) :
        64  unidirectionnel   total 0.820  chauffage 0.810   val_loss 0.1855   arrêt ép. 4
        128 unidirectionnel   total 0.823  chauffage 0.805   val_loss 0.1853   arrêt ép. 2
        128 BIDIRECTIONNEL    total 0.834  chauffage 0.835   val_loss 0.1711   arrêt ép. 17
    Grossir seul n'apporte rien : le plafond n'était pas la capacité mais l'information
    disponible. En unidirectionnel le réseau épuise le passé en 4 époques ; dans les deux
    sens il continue d'apprendre jusqu'à la 17e. Coût : ~6,6x plus lent.

    TÊTE ADDITIVE vs MULTIPLICATIVE — mesuré le 04/08 (503 bâtiments, S = 54, graine 42) :
                             total   chauffage    clim   eau_chaude
        additive             0.836     0.815      0.883     0.822
        multiplicative       0.822     0.788      0.885     0.835
    La multiplicative fait prédire une FORME sans dimension (conso / moyenne annuelle),
    remise à l'échelle par le niveau annuel LightGBM. Elle échoue, et la dégradation suit
    EXACTEMENT l'hétérogénéité de la forme exigée — p90/p10 de la forme max sur le parc :
        eau_chaude  2.3  -> +0.013      clim       2.5  -> +0.002
        total       2.3  -> -0.014      chauffage 13.5  -> -0.027
    Raison : en additif le réseau apprend une relation LOCALE et physique (T_ext = 5 °C,
    UA = 360 -> ~1 kWh/h), identique en Floride et dans le Minnesota. En multiplicatif il
    doit prédire un rapport à la moyenne annuelle, donc connaître le nombre d'heures de
    chauffe de l'année — une propriété GLOBALE du climat :
        HDD18  <500 :  673 h de chauffe/an -> 69 % de l'annuel en 1 semaine, forme p99 28.8
        HDD18 >4000 : 5229 h de chauffe/an ->  7 % de l'annuel en 1 semaine, forme p99  3.8
    Or log(HDD18) explique 72 % de cette amplitude et HDD18 n'est PAS dans s — seule la
    latitude y figure, et elle n'en explique que 57 %.
    À RE-TESTER une fois le climat ajouté à s, pas avant.
    """
    def __init__(self, n_time, n_static, hidden=128, n_out=4, tete=None):
        super().__init__()
        self.tete = tete if tete is not None else TETE
        self.gru  = nn.GRU(n_time, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2 + n_static, hidden), nn.ReLU(),   # *2 : sens avant + arrière
            nn.Linear(hidden, n_out),
            nn.Softplus(),                        # une conso ne peut pas être négative
        )
        if self.tete == 'multiplicative':         # en additif on garde l'init PyTorch d'origine,
            nn.init.constant_(self.head[-2].bias, BIAIS_UN)   # sinon ce n'est plus le modèle du 31/07

    def forward(self, x_time, static, niveau=None):
        """niveau : (B, n_out), ignoré en tête additive.

        En multiplicatif il est fourni DANS L'ÉCHELLE DES CIBLES normalisées (kWh/h divisés
        par ys), pour que la sortie reste comparable à Yn et la val_loss lisible d'une
        version à l'autre.
        """
        h, _ = self.gru(x_time)                           # (B, T, hidden*2)
        s = static.unsqueeze(1).expand(-1, h.size(1), -1) # répète s sur le temps
        out = self.head(torch.cat([h, s], dim=-1))        # (B, T, n_out)
        if self.tete == 'multiplicative':
            out = out * niveau.unsqueeze(1)
        return out

## Partie 1 — Validation du squelette (données factices)

In [ ]:
torch.manual_seed(0)
B, N_TIME = 256, 27
x_time = torch.randn(B, SEQ_LEN, N_TIME)
static = torch.randn(B, N_STATIC)
niveau = torch.ones(B, N_OUT)      # niveau = 1 -> la tête multiplicative est neutre : on teste
                                   # bien le squelette GRU + fusion, pas la mise à l'échelle
y = x_time @ torch.randn(N_TIME, N_OUT) + static.unsqueeze(1) + 0.1 * torch.randn(B, SEQ_LEN, N_OUT)

model = LoadNet(N_TIME, N_STATIC, HIDDEN, N_OUT).to(device)
opt, lossf = torch.optim.Adam(model.parameters(), lr=1e-3), nn.MSELoss()
x_time, static, niveau, y = x_time.to(device), static.to(device), niveau.to(device), y.to(device)
for epoch in range(200):
    opt.zero_grad(); loss = lossf(model(x_time, static, niveau), y); loss.backward(); opt.step()
    if epoch % 40 == 0:
        print(f'epoch {epoch:3d}  loss {loss.item():.4f}')
print('sortie :', model(x_time, static, niveau).shape)

## Partie 2 — Vraies données
Par bâtiment : `f(t)` = météo (5) + schedules d'usage (16) + **consignes de thermostat (2)** + calendrier (6) + **écarts à la consigne (2)** = **31**, en horaire.

Schedules absents (appareil non présent / logement vacant) → 0.
Les consignes sont vides dans la release OEDI : elles sont **reconstruites** à partir des caractéristiques statiques par `inject_setpoints` (cf. `extraction_timeseries_oedi`, section 1bis).

In [ ]:
WEA = ['out.outdoor_air_drybulb_temp..c', 'out.outdoor_air_relative_humidity..percentage',
       'out.weather.wind_speed..meter_per_second',
       'out.weather.direct_normal_solar_radiation..watt_per_m2',
       'out.weather.diffuse_solar_radiation..watt_per_m2']
SCHED = ['out.schedules.' + s for s in [
    'occupants', 'vacancy', 'lighting_interior', 'lighting_garage', 'plug_loads_other',
    'plug_loads_tv', 'clothes_dryer', 'clothes_washer', 'dishwasher', 'cooking_range',
    'ceiling_fan', 'hot_water_fixtures', 'hot_water_clothes_washer', 'hot_water_dishwasher',
    'no_space_cooling', 'no_space_heating']]
# consignes de thermostat : vides à la source, reconstruites par extraction_timeseries_oedi
SETP = ['out.schedules.heating_setpoint..c', 'out.schedules.cooling_setpoint..c']
TGT = ['out.electricity.' + t + '.energy_consumption..kwh' for t in
       ['total', 'heating', 'cooling', 'hot_water']]

# Testé le 31/07 : ajouter 10 colonnes « puissance par appareil » (schedule × niveau annuel
# prédit hors échantillon, cf. static_preds_usages.parquet) n'apporte RIEN — A/B sur les mêmes
# 503 bâtiments : total -0.003, chauffage -0.012, clim -0.003, eau_chaude +0.005, val_loss
# identique. Le GRU reconstruisait déjà le talon à partir des schedules et du vecteur statique.

def load_building(bldg_id, state):
    """Timeseries -> f(t) horaire (météo + schedules + consignes + écarts + calendrier) et cibles."""
    path = DATA / f'{bldg_id}-0.parquet'
    ts = pd.read_parquet(path if path.exists() else f'{TS_BASE}/state={state}/{bldg_id}-0.parquet')
    if not path.exists():
        ts.to_parquet(path)
    ts['timestamp'] = pd.to_datetime(ts['timestamp'])
    brut = ts.set_index('timestamp').reindex(columns=WEA + SCHED + SETP + TGT)

    # Données au pas de 15 min -> horaire. L'agrégation N'EST PAS la même selon la nature :
    #   températures, vitesses, taux d'usage : grandeurs instantanées  -> moyenne
    #   consommations en kWh              : énergies, ça s'additionne -> somme
    # Vérifié sur le bâtiment 65263 : somme des 35 040 pas = 39 484 kWh = l'annuel de
    # metadata_clean. Avec .mean() on obtenait 9 871 kWh, soit 4x trop peu.
    h = brut[WEA + SCHED + SETP].resample('1h').mean().iloc[:8760]
    y = brut[TGT].resample('1h').sum().iloc[:8760]
    h[SCHED] = h[SCHED].fillna(0.0)

    # Les consignes n'existent pas dans la release brute : si elles sont vides ici, c'est que le
    # fichier local date d'avant l'injection (ou vient d'être téléchargé directement depuis OEDI).
    if h[SETP].isna().any().any():
        raise ValueError(
            f'bâtiment {bldg_id} : consignes absentes. Relancer extraction_timeseries_oedi '
            '(Partie A, section 3) avec FORCE = True.')

    i = h.index
    cal = pd.DataFrame({
        'h_sin': np.sin(2*np.pi*i.hour/24),      'h_cos': np.cos(2*np.pi*i.hour/24),
        'd_sin': np.sin(2*np.pi*i.dayofweek/7),  'd_cos': np.cos(2*np.pi*i.dayofweek/7),
        'm_sin': np.sin(2*np.pi*(i.month-1)/12), 'm_cos': np.cos(2*np.pi*(i.month-1)/12),
    }, index=i)

    # Écarts à la consigne = besoin thermique instantané (degrés-heures). La consigne seule ne
    # déclenche rien ; c'est son écart à la température extérieure qui appelle le chauffage/la clim.
    t_ext = h['out.outdoor_air_drybulb_temp..c']
    ecart = pd.DataFrame({
        'ecart_chauffage': (h[SETP[0]] - t_ext).clip(lower=0),   # besoin de chauffe
        'ecart_clim':      (t_ext - h[SETP[1]]).clip(lower=0),   # besoin de froid
    }, index=i)

    # 5 météo + 16 schedules + 2 consignes + 6 calendrier + 2 écarts = 31 entrées
    return pd.concat([h[WEA + SCHED + SETP], cal, ecart], axis=1), y

In [ ]:
# Bâtiments téléchargés (produits par extraction_timeseries_oedi, Partie A)
# ATTENTION : nn_buildings.csv est GROUPÉ PAR ÉTAT. Un `[:N]` ne réduit donc pas seulement la
# taille du parc, il en change le CLIMAT. Les 30 premiers = 22 FL + 6 TX + 2 GA : un parc
# uniquement chaud, où le chauffage moyen tombe à 1 387 kWh/an contre 6 556 sur les 503, soit
# 7 % du total au lieu de 30 %. Le réseau n'a alors presque plus de chauffage à apprendre, et
# le R² « chauffage » est calculé sur une cible quasi nulle — il ne veut plus dire grand-chose.
# Mesuré le 03/08 avec `[:30]` : total 0.558, chauffage 0.602, contre 0.834 / 0.835 le 31/07
# sur les 503. S'ajoute la mémorisation : 24 bâtiments d'entraînement pour un s de 54 colonnes,
# alors que la note plus bas rappelle que ça dégradait déjà à 153.
# Pour itérer vite, TIRER AU HASARD — jamais une tranche.
N_BAT = None          # None = les 503 ; un entier = échantillon aléatoire

_parc = pd.read_csv(DATA / 'nn_buildings.csv')
if N_BAT is not None:
    _parc = _parc.sample(n=N_BAT, random_state=0)
BUILDINGS = list(_parc.itertuples(index=False, name=None))
print(f'parc : {len(BUILDINGS)} bâtiments | {_parc["in.state"].nunique()} états')

def build_dataset(buildings, L=SEQ_LEN):
    Xs, Ys, bids = [], [], []
    for bid, st in buildings:
        f, y = load_building(bid, st)
        n = len(f) // L
        Xs.append(f.values[:n*L].reshape(n, L, -1))
        Ys.append(y.values[:n*L].reshape(n, L, -1))
        bids += [bid] * n
    return np.concatenate(Xs), np.concatenate(Ys), np.array(bids)

X_time, Y, bids = build_dataset(BUILDINGS)
print('X_time', X_time.shape, '| Y', Y.shape, '| fenêtres', len(bids))

In [ ]:
# s = prédictions LightGBM (niveau annuel) + description physique du bâtiment
preds = pd.read_parquet(DATA / 'static_preds.parquet').set_index('bldg_id')
feat  = pd.read_parquet(DATA / 'X_47features.parquet')          # 5 agrégats enveloppe + DSE + 41 autres
feat  = feat.assign(tau=feat['C'] / (feat['UA'] + feat['H_ve']) / 3.6)   # constante de temps (h)
assert set(bids) <= set(feat.index), 'bâtiments absents du sous-ensemble de lgbm_electricity_5features'

# --- Consignes de thermostat : retirées de s ----------------------------------------------
# Les 6 colonnes (2 bases + 2 has_offset + 2 offset_magnitude) sont REDONDANTES ici : f(t)
# contient déjà les deux consignes heure par heure, plus les deux écarts qui en dérivent.
# Elles restent dans X_47features.parquet, car LightGBM en a besoin — `in.heating_setpoint` y
# est la 3e feature sur 108 (gain 8.99) précisément parce qu'il n'a AUCUNE série temporelle.
# Le réseau, lui, les voit deux fois.
# Bénéfice annexe : plus rien à corriger côté statique pour les 108 bâtiments arbitrés par la
# règle du guide ResStock p.135 (cf. inject_setpoints) — seul f(t) était à réparer.
# Au passage, leur encodage était faux : les offset_magnitude sont convertis avec la formule
# ABSOLUE ((F-32)x5/9) au lieu de la formule d'ÉCART (Fx5/9), d'où des « magnitudes » de
# -17.8 °C. Numériquement inoffensif (c'est une translation constante, absorbée par la
# standardisation comme par les seuils d'arbre), mais illisible dans les tableaux de reports/.
# A/B à faire : redondant ne veut pas dire inutile — la tête lit s directement à chaque pas de
# temps, alors que le niveau de consigne présent dans f(t) doit survivre à la récurrence du GRU.
DROP_CONSIGNES = [c for c in feat.columns if 'setpoint' in c]            # 6 colonnes
feat = feat.drop(columns=DROP_CONSIGNES)                                 # 48 -> 42

# --- Présence de l'équipement ------------------------------------------------------------
# 79/503 bâtiments (16 %) ont un chauffe-eau NON électrique et 26/503 (5 %) n'ont pas de clim :
# leur consommation de cet usage est strictement nulle toute l'année. Or `in.water_heater_fuel`
# et `in.hvac_cooling_type` ont été retirés des 48 features par DROP_DHW dans
# lgbm_electricity_5features — le réseau n'avait donc aucun moyen de le savoir.
equip = pd.read_parquet(DATA / 'metadata_clean.parquet',
                        columns=['bldg_id', 'in.hvac_cooling_type', 'in.water_heater_fuel']
                        ).set_index('bldg_id')
a_clim = (equip.loc[bids, 'in.hvac_cooling_type'] != 'None').values.astype('float32')
ecs_el = (equip.loc[bids, 'in.water_heater_fuel'] == 'Electricity').values.astype('float32')

S_brut = np.hstack([preds.loc[bids].values, feat.loc[bids].values,
                    a_clim[:, None], ecs_el[:, None]]).astype('float32')   # 4 + 42 + 2 = 48

# Masque appliqué aux sorties : [total, chauffage, clim, eau_chaude].
# Pas d'équipement -> sortie forcée à zéro. C'est une contrainte physique, pas quelque chose à
# apprendre. Indispensable car Softplus = log(1+e^x) tend vers 0 sans jamais l'atteindre : même
# parfaitement informé, le réseau laisserait un résidu positif sur ces bâtiments.
# `total` n'est pas masqué : il reste non nul même sans clim ni eau chaude électrique.
un = np.ones_like(a_clim)
GATE = np.stack([un, un, a_clim, ecs_el], axis=-1).astype('float32')       # (n_fenêtres, 4)

# Historique : ajouter les 48 features DÉGRADAIT le modèle le 29/07, quand le parc faisait
# 153 bâtiments (122 en entraînement) — le réseau mémorisait. À 503 bâtiments (402 en
# entraînement) il améliore : total +0.031, chauffage +0.044, val_loss -13 %.
# Retirer les 6 consignes remonte le ratio bâtiments d'entraînement / colonnes de s de 7.4 à
# 8.4 (cf. etude_parc_503, §4.4), donc dans le bon sens vis-à-vis de la mémorisation.

# split PAR BÂTIMENT : val = 20 % des bâtiments (jamais vus à l'entraînement)
rng   = np.random.default_rng(42)
uniq  = np.unique(bids)
val_b = set(rng.choice(uniq, size=max(1, round(0.2 * len(uniq))), replace=False))
val   = np.array([b in val_b for b in bids]); tr = ~val

# standardisation ajustée sur le TRAIN uniquement
def fit_std(a):
    ax = tuple(range(a.ndim - 1))
    return a.mean(ax, keepdims=True), a.std(ax, keepdims=True) + 1e-8

# IMPORTANT : on écrit dans de NOUVELLES variables (Xn, Sn) au lieu d'écraser X_time et S_brut.
# Auparavant cette cellule faisait `X_time = (X_time - xm) / xs` : la réexécuter sans relancer
# la cellule précédente normalisait une deuxième fois, et le modèle recevait des entrées
# incohérentes (symptôme : surestimation massive, R² qui s'effondre vers 0.5).
xm, xs = fit_std(X_time[tr]); Xn = (X_time - xm) / xs      # entrées : centrées-réduites
sm, ss = fit_std(S_brut[tr]); Sn = (S_brut - sm) / ss      # idem (gère UA~360, C~28000, binaires)

# cibles : mise à l'échelle SANS centrage -> Yn reste >= 0, compatible Softplus
ys = Y[tr].std(tuple(range(Y.ndim - 1)), keepdims=True) + 1e-8
Yn = Y / ys                                                # Y garde les kWh d'origine (pour l'éval)

# --- Niveau annuel : facteur d'échelle de la tête multiplicative --------------------------
# Utilisé uniquement si TETE == 'multiplicative' (rejetée le 04/08, cf. docstring de LoadNet).
# Calculé quand même pour que le re-test coûte une ligne, et sauvegardé dans le checkpoint.
assert list(preds.columns) == ['total', 'chauffage', 'clim', 'eau_chaude'], \
    f'ordre des colonnes de static_preds incompatible avec TGT : {list(preds.columns)}'

# PLANCHER. LightGBM n'est pas contraint et produit des annuels négatifs ou quasi nuls : sur
# les 503 bâtiments, 18 négatifs sur chauffage (min -695 kWh/an), 16 sur clim, 45 sur ECS.
# Un niveau négatif retournerait la courbe ; un niveau quasi nul l'écraserait DÉFINITIVEMENT.
# L'erreur est asymétrique :
#   plancher trop BAS  -> irrécupérable. À 1e-3 kWh/h (~9 kWh/an), les 18 bâtiments concernés
#                         (chauffage réel médian 135 kWh/an, max 963) exigeraient une forme
#                         soutenue de 15 à 110 pour rattraper.
#   plancher trop HAUT -> rattrapable, Softplus descend aussi bas qu'on veut.
# D'où 1 % de la médiane du parc (train), par usage : la forme maximale requise retombe à 21,
# et les bâtiments planchés sur clim/ECS ont de toute façon un vrai annuel nul et sont gatés.
PLANCHER = 0.01 * np.median(preds.loc[bids].values[tr], axis=0) / 8760.0     # kWh/h, par usage
NIV  = np.maximum(preds.loc[bids].values / 8760.0, PLANCHER).astype('float32')   # (n_fenêtres, 4)
NIVn = (NIV / ys.reshape(1, N_OUT)).astype('float32')       # même échelle que Yn

print(f'train {tr.sum()} fenêtres | val {val.sum()} | bâtiments val {len(val_b)}/{len(uniq)}')
print(f'vecteur statique : {Sn.shape[1]} colonnes (4 LightGBM + {feat.shape[1]} features + 2 présence)')
print(f'  consignes retirées de s : {len(DROP_CONSIGNES)} colonnes | '
      f'ratio bâtiments train / colonnes de s = {(len(uniq) - len(val_b)) / Sn.shape[1]:.1f}')
print(f'sans clim : {int((a_clim == 0).sum())} fenêtres | sans ECS électrique : {int((ecs_el == 0).sum())}')
print(f'min(Yn) = {Yn.min():.3f}  (doit être >= 0) | moyenne(Xn) = {Xn.mean():.2e}  (doit être ~0)')

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import copy

to_t = lambda a: torch.tensor(a, dtype=torch.float32)
loader = DataLoader(TensorDataset(to_t(Xn[tr]), to_t(Sn[tr]), to_t(Yn[tr]),
                                  to_t(GATE[tr]), to_t(NIVn[tr])),
                    batch_size=64, shuffle=True)
Xva, Sva, Yva = to_t(Xn[val]).to(device), to_t(Sn[val]).to(device), to_t(Yn[val]).to(device)
Gva, Nva = to_t(GATE[val]).to(device), to_t(NIVn[val]).to(device)

# le masque (B, 4) s'applique à toutes les heures -> unsqueeze(1) diffuse sur les 168
masque = lambda p, g: p * g.unsqueeze(1)

# tailles lues dans les données, jamais codées en dur (s a changé plusieurs fois : 52, 54, 48)
model = LoadNet(Xn.shape[-1], Sn.shape[1], HIDDEN, N_OUT).to(device)
opt, lossf = torch.optim.Adam(model.parameters(), lr=1e-3), nn.MSELoss()
print(f'f(t) = {Xn.shape[-1]} entrées | s = {Sn.shape[1]} colonnes | tête {TETE}')

# loss INCHANGÉE (MSE sur Yn, masquée) : seule la structure du modèle change, sinon l'A/B
# avec les mesures du 31/07 ne voudrait rien dire.
PATIENCE = 8                                   # époques sans progrès tolérées avant l'arrêt
best, best_state, best_ep = float('inf'), None, -1

for epoch in range(40):
    model.train()
    for xb, sb, yb, gb, nb in loader:
        xb, sb, yb = xb.to(device), sb.to(device), yb.to(device)
        gb, nb = gb.to(device), nb.to(device)
        opt.zero_grad()
        lossf(masque(model(xb, sb, nb), gb), yb).backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        vloss = lossf(masque(model(Xva, Sva, Nva), Gva), Yva).item()
    if epoch % 5 == 0:
        print(f'epoch {epoch:3d}  val_loss {vloss:.4f}')

    if vloss < best:                           # meilleur score jusqu'ici -> on met une copie de côté
        best, best_ep = vloss, epoch
        best_state = copy.deepcopy(model.state_dict())
    elif epoch - best_ep >= PATIENCE:          # plus aucun progrès -> inutile de continuer
        print(f'arrêt anticipé à l\'époque {epoch} (aucun progrès depuis {PATIENCE} époques)')
        break

model.load_state_dict(best_state)              # on repart du MEILLEUR réseau, pas du dernier
print(f'meilleure val_loss {best:.4f} (époque {best_ep})')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

# prédictions sur la validation, masquées puis dé-normalisées en kWh
model.eval()
with torch.no_grad():
    pred = masque(model(Xva, Sva, Nva), Gva).cpu().numpy() * ys
true = Y[val]                                    # kWh d'origine
bva  = bids[val]                                 # bâtiment de chaque fenêtre de validation
NOMS = ['total', 'chauffage', 'clim', 'eau_chaude']

print('=== Validation (bâtiments jamais vus) ===')
for i, n in enumerate(NOMS):
    p, t = pred[..., i].ravel(), true[..., i].ravel()
    print(f'{n:11} R²={r2_score(t, p):6.3f}   RMSE={np.sqrt(((p-t)**2).mean()):.3f} kWh/h')

# contrôle : les usages absents doivent être prédits exactement à zéro
for i, (n, g) in enumerate([('clim', GATE[val][:, 2]), ('eau_chaude', GATE[val][:, 3])], start=2):
    absent = g == 0
    if absent.any():
        print(f'  {n:11} {int(absent.sum())} fenêtres sans équipement -> '
              f'max prédit = {pred[absent, :, i].max():.1e} (doit être 0.0)')

# --- Biais de niveau par bâtiment (bloc temporaire, remplacé par la partie 3 du plan) ------
# Le R² ci-dessus est calculé sur toutes les heures de tous les bâtiments CONFONDUS : il est
# dominé par les gros consommateurs et ne voit pas le biais de niveau, qui est justement ce
# que la tête multiplicative corrige. NMBE = écart relatif entre l'annuel reconstitué et
# l'annuel réel, bâtiment par bâtiment. Critère ASHRAE Guideline 14 (horaire) : |NMBE| <= 10 %.
nmbe = {n: [] for n in NOMS}
for b in np.unique(bva):
    k = bva == b
    for i, n in enumerate(NOMS):
        t, p = true[k, :, i].sum(), pred[k, :, i].sum()
        if t > 0:                                # usage absent (gaté) -> pas de NMBE défini
            nmbe[n].append((p - t) / t * 100)

print('\n=== NMBE par bâtiment — biais sur la consommation annuelle reconstituée ===')
print(f"{'usage':11} {'n':>4} {'|NMBE| médian':>14} {'p90':>8} {'part <10%':>11}")
for n in NOMS:
    a = np.abs(nmbe[n])
    print(f'{n:11} {len(a):4d} {np.median(a):13.1f}% {np.percentile(a, 90):7.1f}% '
          f'{(a < 10).mean()*100:10.0f}%')

# réel vs prédit sur une semaine d'un bâtiment de validation
# ATTENTION : w = 0 n'est pas un cas représentatif — c'est la 1re semaine de janvier du
# bâtiment 530669 (Floride, PAC), qui concentre 47 % de son chauffage annuel. La partie 3
# du plan remplacera ce graphe par un triptyque médian / pire / meilleur NMBE.
w = 0
print(f'\nsemaine tracée : bâtiment {bva[w]}, fenêtre {w}')
fig, axes = plt.subplots(2, 2, figsize=(13, 6))
for i, ax in enumerate(axes.ravel()):
    ax.plot(true[w, :, i], label='réel', lw=1.5)
    ax.plot(pred[w, :, i], label='prédit', lw=1.5, alpha=0.8)
    ax.set(title=NOMS[i], xlabel='heure', ylabel='kWh')
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle(f'Réel vs prédit — 1 semaine (validation, bâtiment {bva[w]})', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Sauvegarde pour l'étude de flexibilité (notebooks/07_flexibilite).
# On garde le modèle, les constantes de normalisation ET la règle de masquage : sans elles,
# impossible de présenter une entrée cohérente ni de reproduire les zéros exacts.
COLONNES = list(load_building(*BUILDINGS[0])[0].columns)
uniq_ids = list(dict.fromkeys(bids))            # ordre d'apparition, un par bâtiment
prem     = {b: np.where(bids == b)[0][0] for b in uniq_ids}

torch.save({
    'state_dict': model.state_dict(),
    'archi'   : 'gru_bidirectionnel',
    'tete'    : TETE,             # 'additive' ou 'multiplicative' : change la signature de forward
    'n_time'  : Xn.shape[-1], 'n_static': Sn.shape[1],
    'hidden'  : HIDDEN,       'n_out'   : N_OUT,
    'xm': xm, 'xs': xs,           # normalisation de f(t)
    'sm': sm, 'ss': ss,           # normalisation de s
    'ys': ys,                     # échelle des cibles (dé-normalisation en kWh)
    'colonnes' : COLONNES,        # noms des entrées de f(t), pour les retrouver par nom
    'buildings': BUILDINGS,       # parc utilisé pour cet entraînement
    'val_b'    : sorted(val_b),   # bâtiments de validation, pour rejouer le même découpage
    # masque de présence d'équipement, par bâtiment : [total, chauffage, clim, eau_chaude]
    'gate'     : {int(b): GATE[prem[b]].tolist() for b in uniq_ids},
    # vecteur statique brut par bâtiment, pour reconstruire s sans refaire les jointures
    'statique' : {int(b): S_brut[prem[b]].tolist() for b in uniq_ids},
    # niveau annuel LightGBM en kWh/h. Inutilisé en tête additive, gardé pour pouvoir rejouer
    # la variante multiplicative sans refaire les jointures. Le modèle l'attend divisé par ys.
    'niveau'   : {int(b): NIV[prem[b]].tolist() for b in uniq_ids},
}, DATA / 'loadnet.pt')

print('modèle sauvegardé ->', DATA / 'loadnet.pt')
print(f'  f(t) = {Xn.shape[-1]} entrées | s = {Sn.shape[1]} | {len(BUILDINGS)} bâtiments'
      f' | tête {TETE}')
# ATTENTION : flexibilite.ipynb (07) reconstruit s avec 52 colonnes alors que le checkpoint en
# attend 54 — il est désynchronisé, et ça ne vient pas de ce changement. À reprendre.